In [ ]:
# Parameters
input_data = "results/data/checkpoints/beforefilter_intermediate_empfaenger_dringlichkeit.pq results/data/checkpoints/beforefilter_intermediate_empfaenger.pq results/data/checkpoints/beforefilter_intermediate_empfaenger_immunologie.pq results/data/checkpoints/beforefilter_intermediate_empfaenger_virologie.pq results/data/checkpoints/beforefilter_intermediate_followup_niere.pq results/data/checkpoints/beforefilter_intermediate_followup_niere_medikation.pq results/data/checkpoints/beforefilter_intermediate_organ_entnahme_niere.pq results/data/checkpoints/beforefilter_intermediate_spender_postmortem.pq results/data/checkpoints/beforefilter_intermediate_spender_postmortem_diagnosen.pq results/data/checkpoints/beforefilter_intermediate_spender_postmortem_labor_blutgase.pq results/data/checkpoints/beforefilter_intermediate_spender_postmortem_labor_blutgruppe.pq results/data/checkpoints/beforefilter_intermediate_spender_postmortem_labor_crossmatch.pq results/data/checkpoints/beforefilter_intermediate_spender_postmortem_labor_hla.pq results/data/checkpoints/beforefilter_intermediate_spender_postmortem_labor_klinische_chemie.pq results/data/checkpoints/beforefilter_intermediate_spender_postmortem_labor_mikrobiologie.pq results/data/checkpoints/beforefilter_intermediate_spender_postmortem_labor_pathologie.pq results/data/checkpoints/beforefilter_intermediate_spender_postmortem_labor_toxikologie.pq results/data/checkpoints/beforefilter_intermediate_spender_postmortem_labor_urin.pq results/data/checkpoints/beforefilter_intermediate_spender_postmortem_labor_virologie.pq results/data/checkpoints/beforefilter_intermediate_spender_postmortem_medikation.pq results/data/checkpoints/beforefilter_intermediate_spender_postmortem_monitoring.pq results/data/checkpoints/beforefilter_intermediate_spender_postmortem_untersuchungen.pq results/data/checkpoints/beforefilter_intermediate_transplantation_postop_untersuchung.pq results/data/checkpoints/beforefilter_intermediate_transplantation.pq results/data/checkpoints/beforefilter_intermediate_warteliste_niere.pq"
display_util = "workflow/scripts/display_util.py"
util = "workflow/scripts/util.py"
datanames = "empfaenger_dringlichkeit empfaenger empfaenger_immunologie empfaenger_virologie followup_niere followup_niere_medikation organ_entnahme_niere spender_postmortem spender_postmortem_diagnosen spender_postmortem_labor_blutgase spender_postmortem_labor_blutgruppe spender_postmortem_labor_crossmatch spender_postmortem_labor_hla spender_postmortem_labor_klinische_chemie spender_postmortem_labor_mikrobiologie spender_postmortem_labor_pathologie spender_postmortem_labor_toxikologie spender_postmortem_labor_urin spender_postmortem_labor_virologie spender_postmortem_medikation spender_postmortem_monitoring spender_postmortem_untersuchungen transplantation_postop_untersuchung transplantation warteliste_niere"
output_data = "results/data/checkpoints/targetpop.pq"

In [ ]:
import pandas as pd
from IPython.display import Markdown
import matplotlib_inline
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
import seaborn as sns

sns.set_theme(style="whitegrid")
matplotlib_inline.backend_inline.set_matplotlib_formats("svg")
%matplotlib inline
_ = plt.ioff()
plt.rcParams["figure.figsize"] = (8, 5)

# Target population

Our target population was defined as:

* Recipients of kidney transplantations from deceased donors
 
We excluded:

* Pediatric cases (recipient younger than 18 years at transplantation date)
* Living donors
* Multi-organ patients
* Patients who received multiple transplantations

## Filtering for Non-Pediatric Cases

In [ ]:
allfiles = dict(zip(datanames.split(" "), input_data.split(" ")))

In [ ]:
empfaenger = pd.read_parquet(allfiles["empfaenger"])
transplantation = pd.read_parquet(allfiles["transplantation"])
spender_postmortem = pd.read_parquet(allfiles["spender_postmortem"])
warteliste_niere = pd.read_parquet(allfiles["warteliste_niere"])
hist_rec = {}
hist_trans = {}

hist_rec["ET Recipients"] = empfaenger["recipient_et_id_et"].nunique()
hist_trans["ET Transplantations"] = transplantation["transplant_et_id"].nunique()

In [ ]:
rec_et = empfaenger.loc[
    ~empfaenger["recipient_et_id_et"].isna(), ["recipient_et_id_et", "birthdate"]
]
assert (
    not rec_et["recipient_et_id_et"].duplicated().any()
), "Duplicate entries in the recipients"
# Organ information is only available for ET, otherwise only codes
trans_et = transplantation.loc[
    ~transplantation["operation_date_et"].isna(),
    [
        "recipient_et_id_et",
        "donor_et_id_et",
        "transplant_et_id",
        "operation_date_et",
        "organ",
    ],
]
assert not trans_et["recipient_et_id_et"].isna().any(), "ET recipients with no ID"

In [ ]:
display(Markdown(f"""
The ET data in the `empfaenger` file provided {rec_et.shape[0]} (potential) recipients,
while the ET data entries with a transplantation date in the `transplantation` file provided data on {trans_et.shape[0]} recipients.
The following venn diagramm shows the overlap.
"""))

In [ ]:
f, ax = plt.subplots(1, 1)
f.suptitle("Overlap between 'empfaenger' and 'transplantation'")
venn2(
    [set(rec_et["recipient_et_id_et"]), set(trans_et["recipient_et_id_et"])],
    set_labels=(
        "Potential Recipients",
        "Recipients with transplantation\noperation information",
    ),
    ax=ax,
)
display(f)

We connected the recipient and transplantation entries to calculate the age at the time of transplantation.

In [ ]:
targetpopulation = pd.merge(rec_et, trans_et, how="inner", on="recipient_et_id_et")

In [ ]:
targetpopulation["age"] = (
    targetpopulation["operation_date_et"] - targetpopulation["birthdate"]
) / 365.25

In [ ]:
hist_rec["Recipients with req. Information"] = targetpopulation[
    "recipient_et_id_et"
].nunique()
hist_trans["Transplantations with req. Information"] = targetpopulation[
    "transplant_et_id"
].nunique()

In [ ]:
f, ax = plt.subplots(1, 1)
sns.ecdfplot(data=targetpopulation, x="age", ax=ax)
ax.set_xlabel("Age at the time of Transplantation")
ax.set_ylabel("ECDF P(Age<=x)")
f.suptitle("Age Distribution Before Filtering")
display(f)

In [ ]:
agefilter = targetpopulation["age"] < 18
display(Markdown(f"""
Of the {targetpopulation.shape[0]} transplantations we discarded {agefilter.sum()} ({agefilter.sum()/targetpopulation.shape[0]*100:.2f}%), as the recipient was younger than 18 years at the time of the operation.
"""))
targetpopulation = targetpopulation.loc[~agefilter,]

In [ ]:
hist_rec["Recipients above 18"] = targetpopulation["recipient_et_id_et"].nunique()
hist_trans["Transplantations above 18"] = targetpopulation["transplant_et_id"].nunique()

## Filtering For Kidney Transplantations

Our data export was limited to kidney data, this filtering process however is not performed within each file. We excluded recipients, who had another organ transplantation.

In [ ]:
f, ax = plt.subplots(1, 1)
sns.countplot(data=targetpopulation, x="organ", ax=ax)
ax.set_xlabel("Organ Transplanted")
ax.set_ylabel("Count")
f.suptitle("Transplanted Organ Distribution")
f.autofmt_xdate(rotation=45)
display(f)

In [ ]:
removeme = targetpopulation.loc[
    ~targetpopulation["organ"].str.endswith("Kidney"), "recipient_et_id_et"
].drop_duplicates()
display(Markdown(f"""
Of the {targetpopulation["recipient_et_id_et"].nunique()} recipients we discarded {removeme.shape[0]} ({removeme.shape[0]/targetpopulation["recipient_et_id_et"].nunique():.2%}), as they had not kidney transplantations.
"""))
targetpopulation = targetpopulation.loc[
    ~targetpopulation["recipient_et_id_et"].isin(removeme),
]

In [ ]:
hist_rec["Kidney Only"] = targetpopulation["recipient_et_id_et"].nunique()
hist_trans["Kidney Only"] = targetpopulation["transplant_et_id"].nunique()

## Filtering for single transplantations

Our data export was limited to kidney data, this filtering process however is not performed within each file. We excluded recipients, who had another organ transplantation.

In [ ]:
occurences = targetpopulation["recipient_et_id_et"].value_counts()
f, ax = plt.subplots(1, 1)
sns.countplot(x=occurences, ax=ax)
ax.set_xlabel("Number of transplantations")
ax.set_ylabel("Count of Patients")
f.suptitle("Repeat transplantation")
f.autofmt_xdate(rotation=45)
display(f)

In [ ]:
removeme = occurences.index[occurences > 1]
display(Markdown(f"""
Of the {targetpopulation["recipient_et_id_et"].nunique()} recipients we discarded {removeme.shape[0]} ({removeme.shape[0]/targetpopulation["recipient_et_id_et"].nunique():.2%}), as they received multiple transplantations.
"""))
targetpopulation = targetpopulation.loc[
    ~targetpopulation["recipient_et_id_et"].isin(removeme),
]

In [ ]:
hist_rec["No Repeats"] = targetpopulation["recipient_et_id_et"].nunique()
hist_trans["No Repeats"] = targetpopulation["transplant_et_id"].nunique()

The column `number_of_any_transplants`also contains information on transplantations, which recipients had, but which are not listed in the registry.

In [ ]:
multiple = (
    warteliste_niere.query("number_of_any_transplants > 1")
    .dropna(how="all")
    .dropna(how="all", axis=1)["recipient_et_id_et"]
    .drop_duplicates()
)

In [ ]:
removeme = multiple[multiple.isin(targetpopulation["recipient_et_id_et"])]
display(Markdown(f"""
Of the {targetpopulation["recipient_et_id_et"].nunique()} recipients we discarded {removeme.shape[0]} ({removeme.shape[0]/targetpopulation["recipient_et_id_et"].nunique():.2%}), as they received multiple transplantations (according to the waiting list data).
"""))
targetpopulation = targetpopulation.loc[
    ~targetpopulation["recipient_et_id_et"].isin(removeme),
]

In [ ]:
hist_rec["No prev. Trans."] = targetpopulation["recipient_et_id_et"].nunique()
hist_trans["No prev. Trans."] = targetpopulation["transplant_et_id"].nunique()

## Filtering For Only Deceased Donors

Only donors for which ET, IQTIG or DSO provided data in the `spender_postmortem` file were kept.

In [ ]:
spender = (
    pd.concat(
        [
            spender_postmortem["donor_et_dso"],
            spender_postmortem["donor_et_id_et"],
            spender_postmortem["donor_et_iqtig"],
        ]
    )
    .dropna()
    .drop_duplicates()
)

In [ ]:
f, ax = plt.subplots(1, 1)
f.suptitle("Overlap between filtered transplantations and 'spender_postmortem'")
venn2(
    [set(spender), set(targetpopulation["donor_et_id_et"])],
    set_labels=("Filtered transplantations donors", "Deceased donors"),
    ax=ax,
)
display(f)

In [ ]:
targetpopulation = targetpopulation[
    targetpopulation["donor_et_id_et"].isin(spender)
].drop(columns=["birthdate", "age", "organ"])
# TODO What is with living donors, who also donated after death, we need another filter!!!

In [ ]:
display(Markdown(f"""
After the filtering process {targetpopulation.shape[0]} transplantations are considered as our target population.
"""))

In [ ]:
hist_rec["Deceased Only"] = targetpopulation["recipient_et_id_et"].nunique()
hist_trans["Deceased Only"] = targetpopulation["transplant_et_id"].nunique()

In [ ]:
# Convert hist_rec and hist_trans to DataFrames for easier plotting
hist_rec_df = pd.DataFrame(list(hist_rec.items()), columns=["Stage", "Count"])
hist_trans_df = pd.DataFrame(list(hist_trans.items()), columns=["Stage", "Count"])

# Create subplots
fig, axs = plt.subplots(2, 1, figsize=(10, 10))

# Plot hist_rec
axs[0].plot(
    hist_rec_df["Stage"],
    hist_rec_df["Count"],
    marker="o",
    linestyle="-",
    label="Recipients",
)
axs[0].set_xlabel("Stage")
axs[0].set_ylabel("Count")
axs[0].set_title("Recipients Over Stages")
axs[0].legend()
axs[0].tick_params(axis="x", rotation=45)
axs[0].grid(True)

# Add value labels for hist_rec
for i, count in enumerate(hist_rec_df["Count"]):
    axs[0].annotate(
        f"{count}", (i, count), textcoords="offset points", xytext=(0, 5), ha="center"
    )

# Plot hist_trans
axs[1].plot(
    hist_trans_df["Stage"],
    hist_trans_df["Count"],
    marker="o",
    linestyle="-",
    label="Transplantations",
)
axs[1].set_xlabel("Stage")
axs[1].set_ylabel("Count")
axs[1].set_title("Transplantations Over Stages")
axs[1].legend()
axs[1].tick_params(axis="x", rotation=45)
axs[1].grid(True)

# Add value labels for hist_trans
for i, count in enumerate(hist_trans_df["Count"]):
    axs[1].annotate(
        f"{count}", (i, count), textcoords="offset points", xytext=(0, 5), ha="center"
    )

fig.tight_layout()
display(fig)

In [ ]:
targetpopulation = (
    targetpopulation.rename(columns={"operation_date_et": "recipient_op_date"})
    .sort_values(["recipient_et_id_et", "recipient_op_date"])
    .set_index(
        [
            "transplant_et_id",
            "recipient_et_id_et",
            "donor_et_id_et",
            "recipient_op_date",
        ]
    )
)
targetpopulation["recipient_transplant_running_id"] = (
    targetpopulation.index.get_level_values("recipient_et_id_et")
    .to_series()
    .duplicated()
    .groupby("recipient_et_id_et")
    .cumsum()
    + 1
).set_axis(targetpopulation.index, axis=0)

In [ ]:
targetpopulation.reset_index().to_parquet(output_data)